## **Projeto:** Merca Data Platform

##**Squad:** 2 | Camada Bronze
### Objetivo
Ingerir os dados brutos do catálogo de produtos na camada Bronze do Delta Lake.
Nenhuma transformação é aplicada ao conteúdo — apenas colunas de auditoria e particionamento temporal são adicionados para otimizar consultas futuras.
### Origem e Destino
| **Origem** | `real-time-data/<snapshot>/ecommerce_produtos.parquet` (ADLS) |
| **Destino** | `squad2/bronze/ecommerce_produtos` (Delta Lake) |
| **Checkpoint** | `squad2/control/ecommerce_produtos/control_file.json` |
| **Modo de escrita** | `append` incremental por snapshot |
| **Particionamento** | `ingestion_year / ingestion_month / ingestion_day / ingestion_hour` |
### Colunas de Auditoria Adicionadas
| Coluna | Descrição |
| `bronze_source_file` | Caminho completo do arquivo Parquet de origem |
| `bronze_ingested_at` | Timestamp de ingestão na Bronze |
| `_source` | Fonte dos dados (`real-time-data`) |
| `_camada` | Camada atual (`bronze`) |
| `ingestion_year` | Ano de ingestão (usado como partição) |
| `ingestion_month` | Mês de ingestão (usado como partição) |
| `ingestion_day` | Dia de ingestão (usado como partição) |
| `ingestion_hour` | Hora de ingestão (usado como partição) |
### Dependências
| Notebook | Motivo |
| `feat_squad2_99_helpers` | Conexão ADLS, leitura Parquet, escrita Delta, checkpoint e log |

In [0]:
%run ../utils/feat_squad2_99_helpers

In [0]:
import logging
from pyspark.sql.functions import lit, current_timestamp, year, month, dayofmonth, hour
 
logging.getLogger("azure").setLevel(logging.WARNING)
 
TABELA = "ecommerce_produtos"
CAMADA = "bronze"
 
inicio = log_inicio(f"feat_squad2_{CAMADA}_{TABELA}")
log.info(f"Tabela : {TABELA}")
log.info(f"Camada : {CAMADA}")
log.info(f"Path   : {get_delta_path(CAMADA, TABELA)}")

In [0]:
def processar_snapshot(snapshot_id: str) -> bool:
   
    try:
        # Leitura do parquet
        df = ler_parquet(snapshot_id, TABELA)
        
        # Construir caminho do arquivo fonte
        source_file = "real-time-data/" + snapshot_id + "/" + TABELA + ".parquet"
 
        # Adicionar colunas de auditoria e partições
        df_bronze = df \
            .withColumn("bronze_source_file", lit(source_file)) \
            .withColumn("bronze_ingested_at", current_timestamp()) \
            .withColumn("_source",         lit("real-time-data")) \
            .withColumn("_camada",         lit(CAMADA)) \
            .withColumn("ingestion_year",  year(current_timestamp()).cast("string")) \
            .withColumn("ingestion_month", month(current_timestamp()).cast("string")) \
            .withColumn("ingestion_day",   dayofmonth(current_timestamp()).cast("string")) \
            .withColumn("ingestion_hour",  hour(current_timestamp()).cast("string"))
 
        # Gravar em Delta Lake
        sucesso = gravar_delta(df_bronze, CAMADA, TABELA)
        return sucesso
 
    except Exception as e:
        log.error(f"Erro ao processar {snapshot_id}: {str(e)}")
        return False

In [0]:
snapshots   = sorted(listar_snapshots())
processados = ler_checkpoint(CAMADA, TABELA)
novos       = [s for s in snapshots if s not in processados]
 
if not novos:
    log.info("Bronze " + TABELA + " em dia - nenhum snapshot novo.")
else:
    log.info(str(len(novos)) + " snapshot(s) novo(s) encontrado(s).")
    salvar_checkpoint(CAMADA, TABELA, processados, status="PROCESSANDO")
 
    for snapshot_id in novos:
        log.info("Processando: " + snapshot_id)
        sucesso = processar_snapshot(snapshot_id)
        if sucesso:
            processados.add(snapshot_id)
            log.info("OK: " + snapshot_id)
        else:
            log.warning("FALHOU: " + snapshot_id + " - sera retentado no proximo ciclo.")
 
    salvar_checkpoint(CAMADA, TABELA, processados, status="CONCLUIDO")
    log.info("Bronze " + TABELA + " concluida.")
 
log_fim("feat_squad2_" + CAMADA + "_" + TABELA, inicio)

### Validação Pontual
Execute esta célula isoladamente para verificar o estado atual sem iniciar o loop contínuo.

In [0]:
try:
    from deltalake import DeltaTable
    
    dt = DeltaTable(
        get_delta_path(CAMADA, TABELA),
        storage_options=get_storage_options()
    )
    count = dt.to_pyarrow_dataset().count_rows()
    status = ler_status_checkpoint(CAMADA, TABELA)
    processados = len(ler_checkpoint(CAMADA, TABELA))
    
    print("\n" + "=" * 70)
    print(f"VALIDAÇÃO — {TABELA.upper()}")
    print("=" * 70)
    print(f"Linhas gravadas : {count:,}")
    print(f"Snapshots proc. : {processados}")
    print(f"Status          : {status}")
    print("=" * 70)
 
except Exception as e:
    log.error(f"Erro na validação: {str(e)}")

### Polling Loop
Inicia o monitoramento contínuo da Bronze.
**Bronze não tem dependência de camada anterior.**
Para encerrar, interrompa a execução manualmente.

In [0]:
snapshots   = sorted(listar_snapshots())
processados = ler_checkpoint(CAMADA, TABELA)
novos       = [s for s in snapshots if s not in processados]

if not novos:
    log.info("Bronze " + TABELA + " em dia - nenhum snapshot novo.")
else:
    log.info(str(len(novos)) + " snapshot(s) novo(s) encontrado(s).")
    salvar_checkpoint(CAMADA, TABELA, processados, status="PROCESSANDO")

    for snapshot_id in novos:
        log.info("Processando: " + snapshot_id)
        sucesso = processar_snapshot(snapshot_id)
        if sucesso:
            processados.add(snapshot_id)
            log.info("OK: " + snapshot_id)
        else:
            log.warning("FALHOU: " + snapshot_id + " - sera retentado no proximo ciclo.")

    salvar_checkpoint(CAMADA, TABELA, processados, status="CONCLUIDO")
    log.info("Bronze " + TABELA + " concluida.")

log_fim("feat_squad2_" + CAMADA + "_" + TABELA, inicio)

In [0]:
import logging
from pyspark.sql.functions import lit, current_timestamp, year, month, dayofmonth, hour

logging.getLogger("azure").setLevel(logging.WARNING)

TABELA = "ecommerce_produtos"
CAMADA = "bronze"

inicio = log_inicio(f"feat_squad2_{CAMADA}_{TABELA}")
log.info(f"Tabela : {TABELA}")
log.info(f"Camada : {CAMADA}")

def processar_snapshot(snapshot_id: str) -> bool:
    try:
        df = ler_parquet(snapshot_id, TABELA)
        source_file = "real-time-data/" + snapshot_id + "/" + TABELA + ".parquet"

        df_bronze = df \
            .withColumn("bronze_source_file", lit(source_file)) \
            .withColumn("bronze_ingested_at", current_timestamp()) \
            .withColumn("_source",         lit("real-time-data")) \
            .withColumn("_camada",         lit(CAMADA)) \
            .withColumn("ingestion_year",  year(current_timestamp()).cast("string")) \
            .withColumn("ingestion_month", month(current_timestamp()).cast("string")) \
            .withColumn("ingestion_day",   dayofmonth(current_timestamp()).cast("string")) \
            .withColumn("ingestion_hour",  hour(current_timestamp()).cast("string"))

        sucesso = gravar_delta(df_bronze, CAMADA, TABELA)
        
        if sucesso:
            log.info(f"OK {snapshot_id} → {df_bronze.count()} linhas gravadas.")
        return sucesso

    except Exception as e:
        log.error(f"Erro ao processar {snapshot_id}: {str(e)}")
        return False

# Processa
try:
    snapshots = sorted(listar_snapshots())
    processados = ler_checkpoint(CAMADA, TABELA)
    novos = [s for s in snapshots if s not in processados]
    
    log.info(f"{len(snapshots)} snapshots em raw")
    log.info(f"{len(processados)} snapshots já processados")
    log.info(f"{len(novos)} snapshots NOVOS")
    
    if novos:
        salvar_checkpoint(CAMADA, TABELA, set(), status="PROCESSANDO")
        
        sucesso_count = 0
        for novo in novos:
            if processar_snapshot(novo):
                sucesso_count += 1
                ler_e_salvar_checkpoint(CAMADA, TABELA, novo)
        
        salvar_checkpoint(CAMADA, TABELA, ler_checkpoint(CAMADA, TABELA), status="CONCLUIDO")
        log.info(f"✅ {sucesso_count}/{len(novos)} snapshots processados com sucesso")
    else:
        log.info("✅ Nenhum snapshot novo para processar")

except Exception as e:
    log.error(f"Erro geral: {str(e)}")

log.info("=" * 70)
log.info(f"Bronze {TABELA} concluida!")
log.info("=" * 70)